# Exploración de Datos — Conciliación Bancaria

## Objetivo

Antes de escribir cualquier línea de lógica, es necesario entender la estructura de los archivos CSV que se van a procesar. Este notebook es de gran ayuda para responder la cuestion de:

> ¿Cómo extraemos **tienda** y **lote** de una referencia como `D59900001` o `BD2040026`?

Si nos equivocamos aquí, la conciliación fallaría — emparejando registros
incorrectos sin que nadie se dé cuenta, lo cual evidentemente no queremos.

### Archivos analizados
- `datos/movimientos_banco.csv` — 1272 filas, 14 columnas
- `datos/libro_contable.csv` — 36324 filas, 10 columnas

In [51]:
import pandas as pd
import re

# Cargamos ambos archivos
df_banco = pd.read_csv('../datos/movimientos_banco.csv')
df_libro = pd.read_csv('../datos/libro_contable.csv')

print(f'Banco: {df_banco.shape[0]} filas, {df_banco.shape[1]} columnas')
print(f'Libro: {df_libro.shape[0]} filas, {df_libro.shape[1]} columnas')

Banco: 1272 filas, 14 columnas
Libro: 36324 filas, 10 columnas


## 1. Vistazo inicial al banco

Inspeccionamos las primeras filas para entender la estructura real de los datos,
especialmente las columnas `Referencia` y `Descripción`.

In [52]:
df_banco.head()

,Número de Extracto,Fecha Contable,Estado,Referencia,Fecha Efectiva,Descripción,Tipo,Monto,Banco,Cuenta Bancaria,Cuenta Contable,Sub-cuenta,Código de Transacción,Divisa
0,P-8 2026 10 al 10 30,2026-04-30,UNRECONCILED,D59900001,2026-03-10,LIQ TARJETA DEBITO MAESTRO BDV-LIQ TARJETA DEB...,CREDIT,48343.64,VENEZUELA MERCHANT,1020684710000014766,110108,6,CD-0017425,VES
1,P-8 2026 10 al 10 30,2026-04-30,UNRECONCILED,C10000181,2026-03-10,LIQUIDACION T CREDITO VS MC BD-LIQUIDACION T C...,CREDIT,4864.20,VENEZUELA MERCHANT,1020684710000014766,110108,6,CD-0017430,VES
2,P-8 2026 10 al 10 30,2026-04-30,UNRECONCILED,D10800724,2026-03-10,LIQ TARJETA DEBITO MAESTRO BDV-LIQ TARJETA DEB...,CREDIT,19386.22,VENEZUELA MERCHANT,1020684710000014766,110108,6,CD-0017425,VES
3,P-8 2026 10 al 10 30,2026-04-30,UNRECONCILED,D10800106,2026-03-10,LIQ TARJETA DEBITO MAESTRO BDV-LIQ TARJETA DEB...,CREDIT,6797.41,VENEZUELA MERCHANT,1020684710000014766,110108,6,CD-0017425,VES
4,P-8 2026 10 al 10 30,2026-04-30,UNRECONCILED,D10800635,2026-03-10,LIQ TARJETA DEBITO MAESTRO BDV-LIQ TARJETA DEB...,CREDIT,12771.38,VENEZUELA MERCHANT,1020684710000014766,110108,6,CD-0017425,VES


## 2. Identificación de prefijos

Cada referencia sigue el patrón `[Prefijo][Tienda][Lote]`, tal y como nos comenta la pista en el enunciado. Extraemos las letras
iniciales para identificar todos los medios de pago presentes.

In [53]:
# Extraemos el prefijo alfabético de cada referencia
df_banco['prefijo'] = df_banco['Referencia'].str.extract(r'^([A-Za-z]+)')

print('=== Prefijos en el banco ===')
print(df_banco['prefijo'].value_counts())
print(f'\nReferencias sin prefijo alfabético: {df_banco["prefijo"].isna().sum()}')

=== Prefijos en el banco ===
prefijo
D     917
BD    191
BC     84
M      48
C      30
Name: count, dtype: int64

Referencias sin prefijo alfabético: 2


### Registros sin prefijo alfabético

Hay 2 referencias que comienzan con dígitos. Las inspeccionamos para entender
su naturaleza y decidir cómo tratarlas en el sistema.

In [54]:
sin_prefijo = df_banco[df_banco['prefijo'].isna()]
print(sin_prefijo[['Referencia', 'Descripción', 'Monto', 'Tipo']].to_string())

print("""
\n DECISIÓN DE DISEÑO: Estos 2 registros (01046, 02046) no participan
en la conciliación automática — sin prefijo no es posible extraer tienda
ni lote, por lo tanto no tienen llave de conciliación. A pesar de que en la descripción nos comente que 
se hicieron con una tarjeta de DEBITO, es mejor no asumir para no correr riesgos innecesarios.

Sin embargo, descartarlos sería un error. El sistema los incluye
en el reporte final con estado 'No Conciliado' y una nota que los
identifica como casos sin prefijo, para que el área de Finanzas pueda
investigarlos manualmente.

Estos registros también se contabilizan en el resumen del API bajo
el campo 'sin_prefijo' para que queden visibles.""")

    Referencia                                                    Descripción     Monto   Tipo
702      01046  LIQ TARJETA DEBITO MAESTRO BDV-LIQ TARJETA DEBITO MAESTRO BDV  27024.54  DEBIT
932      02046  LIQ TARJETA DEBITO MAESTRO BDV-LIQ TARJETA DEBITO MAESTRO BDV   2938.69  DEBIT


 DECISIÓN DE DISEÑO: Estos 2 registros (01046, 02046) no participan
en la conciliación automática — sin prefijo no es posible extraer tienda
ni lote, por lo tanto no tienen llave de conciliación. A pesar de que en la descripción nos comente que 
se hicieron con una tarjeta de DEBITO, es mejor no asumir para no correr riesgos innecesarios.

Sin embargo, descartarlos sería un error. El sistema los incluye
en el reporte final con estado 'No Conciliado' y una nota que los
identifica como casos sin prefijo, para que el área de Finanzas pueda
investigarlos manualmente.

Estos registros también se contabilizan en el resumen del API bajo
el campo 'sin_prefijo' para que queden visibles.


## 3. Estructura de longitudes por prefijo

Calculamos la longitud total de cada referencia agrupada por prefijo.
Si la longitud es fija dentro de cada grupo, el parser será determinístico.

In [55]:
df_banco['longitud_ref'] = df_banco['Referencia'].str.len()

print('=== Longitud de referencias por prefijo (banco) ===')
print(df_banco.groupby('prefijo')['longitud_ref'].value_counts().sort_index())

=== Longitud de referencias por prefijo (banco) ===
prefijo  longitud_ref
BC       9                84
BD       9               191
C        9                30
D        9               917
M        7                48
Name: count, dtype: int64


## 4. Identificación visual del patrón Tienda + Lote

Con las longitudes confirmadas, inspeccionamos ejemplos reales para identificar
visualmente dónde termina la tienda y empieza el lote.

In [56]:
for prefijo in ['D', 'C', 'BD', 'BC', 'M']:
    ejemplos = df_banco[df_banco['prefijo'] == prefijo]['Referencia'].head(5).tolist()
    print(f'Prefijo [{prefijo}]: {ejemplos}')

Prefijo [D]: ['D59900001', 'D10800724', 'D10800106', 'D10800635', 'D11500006']
Prefijo [C]: ['C10000181', 'C13700008', 'C13700008', 'C16100151', 'C41600127']
Prefijo [BD]: ['BD4280009', 'BD5030009', 'BD2060009', 'BD3070009', 'BD4840009']
Prefijo [BC]: ['BC4010009', 'BC4840009', 'BC1450009', 'BC2790009', 'BC7010009']
Prefijo [M]: ['M156009', 'M428009', 'M605009', 'M119004', 'M182004']


## 5. Análisis del libro contable

Verificamos si el libro sigue el mismo patrón de prefijos y longitudes que el banco.

In [57]:
df_libro['prefijo'] = df_libro['Número de Transacción'].str.extract(r'^([A-Za-z]+)')
df_libro['longitud_ref'] = df_libro['Número de Transacción'].str.len()

print('=== Prefijos en el libro ===')
print(df_libro['prefijo'].value_counts())
print(f'\nRegistros sin prefijo (serán filtrados): {df_libro["prefijo"].isna().sum()}')

print('\n=== Longitud por prefijo ===')
print(df_libro.groupby('prefijo')['longitud_ref'].value_counts().sort_index())

=== Prefijos en el libro ===
prefijo
BD    401
D     276
M      33
BC     13
C       6
Name: count, dtype: int64

Registros sin prefijo (serán filtrados): 35595

=== Longitud por prefijo ===
prefijo  longitud_ref
BC       9                13
BD       9               401
C        9                 6
D        9               276
M        8                33
Name: count, dtype: int64


## 6. Hallazgo crítico — Inconsistencia en prefijo M

El prefijo `M` (Monedero Patria) tiene longitud **7** en el banco pero **8** en el libro.
Esto requiere análisis específico para entender la diferencia.

In [58]:
ejemplos_banco = df_banco[df_banco['prefijo'] == 'M']['Referencia'].head(5).tolist()
ejemplos_libro = df_libro[df_libro['prefijo'] == 'M']['Número de Transacción'].head(5).tolist()

print(f'M en banco (len=7): {ejemplos_banco}')
print(f'M en libro (len=8): {ejemplos_libro}')

# Descomponemos carácter por carácter para entender la diferencia
print('\n=== Descomposición de M en banco ===')
print(f"{'':<12} {'pos0':<6} {'pos1-3':<8} {'pos4':<6} {'pos5-7'}")
print('-' * 45)
for ref in ejemplos_banco:
    numeros = ref[1:]
    print(f'{ref:<12} {ref[0]:<6} {numeros[:3]:<8} {numeros[3]:<6} {numeros[4:]}')

print('\n=== Descomposición de M en libro ===')
print(f"{'':<12} {'pos0':<6} {'pos1-3':<8} {'pos4':<6} {'pos5-7'}")
print('-' * 45)
for ref in ejemplos_libro:
    numeros = ref[1:]
    print(f'{ref:<12} {ref[0]:<6} {numeros[:3]:<8} {numeros[3]:<6} {numeros[4:]}')

print("""
\n CONCLUSIÓN: El libro agrega un dígito '0' fijo en la posición 4.
   La tienda (3 dígitos) y el lote (3 dígitos) son los mismos en ambos sistemas.
   El parser debe saltar ese dígito extra cuando procesa referencias del libro.""")

M en banco (len=7): ['M156009', 'M428009', 'M605009', 'M119004', 'M182004']
M en libro (len=8): ['M1000001', 'M1560001', 'M1800001', 'M2830001', 'M6000001']

=== Descomposición de M en banco ===
             pos0   pos1-3   pos4   pos5-7
---------------------------------------------
M156009      M      156      0      09
M428009      M      428      0      09
M605009      M      605      0      09
M119004      M      119      0      04
M182004      M      182      0      04

=== Descomposición de M en libro ===
             pos0   pos1-3   pos4   pos5-7
---------------------------------------------
M1000001     M      100      0      001
M1560001     M      156      0      001
M1800001     M      180      0      001
M2830001     M      283      0      001
M6000001     M      600      0      001


 CONCLUSIÓN: El libro agrega un dígito '0' fijo en la posición 4.
   La tienda (3 dígitos) y el lote (3 dígitos) son los mismos en ambos sistemas.
   El parser debe saltar ese dígito extra cua

## 7. Tabla definitiva de parsing

Con toda la información recopilada, definimos la estructura exacta de cada prefijo.

In [59]:
PREFIJO_CONFIG = {
    'D':  {'len_prefijo': 1, 'len_tienda': 3, 'len_lote_banco': 5, 'len_lote_libro': 5},
    'C':  {'len_prefijo': 1, 'len_tienda': 3, 'len_lote_banco': 5, 'len_lote_libro': 5},
    'BD': {'len_prefijo': 2, 'len_tienda': 3, 'len_lote_banco': 4, 'len_lote_libro': 4},
    'BC': {'len_prefijo': 2, 'len_tienda': 3, 'len_lote_banco': 4, 'len_lote_libro': 4},
    'M':  {'len_prefijo': 1, 'len_tienda': 3, 'len_lote_banco': 3, 'len_lote_libro': 3},
}

print(f"{'Prefijo':<8} {'Len prefijo':<14} {'Len tienda':<13} {'Len lote banco':<17} {'Len lote libro'}")
print('-' * 60)
for p, cfg in PREFIJO_CONFIG.items():
    nota = '  ← dígito 0 fijo en pos 4 (se omite)' if p == 'M' else ''
    print(f"{p:<8} {cfg['len_prefijo']:<14} {cfg['len_tienda']:<13} {cfg['len_lote_banco']:<17} {cfg['len_lote_libro']}{nota}")

Prefijo  Len prefijo    Len tienda    Len lote banco    Len lote libro
------------------------------------------------------------
D        1              3             5                 5
C        1              3             5                 5
BD       2              3             4                 4
BC       2              3             4                 4
M        1              3             3                 3  ← dígito 0 fijo en pos 4 (se omite)


## 8. Parser final y validación contra datos reales

Implementamos `parse_reference()` con el parámetro `source` para manejar
la inconsistencia de `M`, y lo validamos contra los 1272 registros reales del banco
y los registros del libro.

In [60]:
def parse_reference(ref: str, source: str = 'banco') -> dict:
    """
    Descompone una referencia en prefijo, tienda y lote.
    
    Parámetros
    ----------
    ref    : str   — código de referencia (ej. 'D59900001')
    source : str   — 'banco' o 'libro' (necesario para manejar prefijo M)
    
    Retorna
    -------
    dict con claves: 'prefijo', 'tienda', 'lote'
    Retorna None en cada campo si la referencia no puede parsearse.
    """
    if not isinstance(ref, str):
        return {'prefijo': None, 'tienda': None, 'lote': None}
    
    match = re.match(r'^([A-Za-z]+)', ref)
    if not match:
        return {'prefijo': None, 'tienda': None, 'lote': None}
    
    prefijo = match.group(1).upper()
    numeros = ref[len(prefijo):]

    if prefijo in ('D', 'C', 'BD', 'BC'):
        tienda = numeros[:3]
        lote   = numeros[3:]
    elif prefijo == 'M':
        tienda = numeros[:3]
        # El libro inserta un '0' fijo en posición 4 que el banco no tiene
        lote   = numeros[3:] if source == 'banco' else numeros[4:]
    else:
        return {'prefijo': prefijo, 'tienda': None, 'lote': None}
    
    return {'prefijo': prefijo, 'tienda': tienda, 'lote': lote}


# ── Validación con casos del enunciado ──────────────────────────────────────
casos = [
    ('D59900001',  'banco'),
    ('C10000181',  'banco'),
    ('BD2040026',  'banco'),
    ('BC4010009',  'banco'),
    ('M156009',    'banco'),
    ('M1560001',   'libro'),
]

print('=== Casos del enunciado ===')
print(f"{'Referencia':<14} {'Source':<8} {'Prefijo':<10} {'Tienda':<10} {'Lote':<8} {'Len lote'}")
print('-' * 58)
for ref, source in casos:
    r = parse_reference(ref, source)
    lote = r['lote'] or ''
    print(f"{ref:<14} {source:<8} {str(r['prefijo']):<10} {str(r['tienda']):<10} {lote:<8} {len(lote)}")

# ── Validación contra datos reales ──────────────────────────────────────────
df_banco['tienda'] = df_banco['Referencia'].apply(lambda r: parse_reference(r, 'banco')['tienda'])
df_banco['lote']   = df_banco['Referencia'].apply(lambda r: parse_reference(r, 'banco')['lote'])

df_libro['tienda'] = df_libro['Número de Transacción'].apply(lambda r: parse_reference(r, 'libro')['tienda'])
df_libro['lote']   = df_libro['Número de Transacción'].apply(lambda r: parse_reference(r, 'libro')['lote'])

print('\n=== Validación banco — longitudes por prefijo ===')
print(f"{'Prefijo':<10} {'Len tienda':<15} {'Len lote':<12} {'Conteo'}")
print('-' * 45)
for p in ['D', 'C', 'BD', 'BC', 'M']:
    s = df_banco[df_banco['prefijo'] == p]
    print(f"{p:<10} {str(s['tienda'].str.len().unique().tolist()):<15} {str(s['lote'].str.len().unique().tolist()):<12} {len(s)}")

print('\n=== Validación libro — longitudes por prefijo ===')
print(f"{'Prefijo':<10} {'Len tienda':<15} {'Len lote':<12} {'Conteo'}")
print('-' * 45)
for p in ['D', 'C', 'BD', 'BC', 'M']:
    s = df_libro[df_libro['prefijo'] == p]
    print(f"{p:<10} {str(s['tienda'].str.len().unique().tolist()):<15} {str(s['lote'].str.len().unique().tolist()):<12} {len(s)}")

=== Casos del enunciado ===
Referencia     Source   Prefijo    Tienda     Lote     Len lote
----------------------------------------------------------
D59900001      banco    D          599        00001    5
C10000181      banco    C          100        00181    5
BD2040026      banco    BD         204        0026     4
BC4010009      banco    BC         401        0009     4
M156009        banco    M          156        009      3
M1560001       libro    M          156        001      3

=== Validación banco — longitudes por prefijo ===
Prefijo    Len tienda      Len lote     Conteo
---------------------------------------------
D          [3]             [5]          917
C          [3]             [5]          30
BD         [3]             [4]          191
BC         [3]             [4]          84
M          [3]             [3]          48

=== Validación libro — longitudes por prefijo ===
Prefijo    Len tienda      Len lote     Conteo
---------------------------------------------
D 

## 9. Conclusiones

| Hallazgo | Decisión |
|----------|----------|
| 5 prefijos identificados: `D`, `C`, `BD`, `BC`, `M` | Tabla de parsing definida por prefijo |
| La tienda siempre ocupa 3 dígitos en todos los prefijos | `numeros[:3]` para tienda en todos los casos |
| El lote varía: 5 dígitos (`D`,`C`), 4 (`BD`,`BC`), 3 (`M`) | `numeros[3:]` para el lote (ajustado por prefijo) |
| Prefijo `M`: el libro inserta un `0` fijo en posición 4 | Parámetro `source='banco'/'libro'` en el parser |
| 2 referencias sin prefijo (`01046`, `02046`) | Se incluyen en el reporte como `No Conciliado` con nota para que Finanzas las investigue manualmente |


### Función lista para producción

`parse_reference(ref, source)` está validada contra los **1272 registros reales** del banco
y los registros con prefijo del libro. Se llevará tal cual a `src/file_processing.py`.